# Notebook 72: V8.8 Annotated-Only Whole-Population Trial

**Colab-safe:** single notebook, no `argparse`, no external Python module, no one-page default.

The notebook inventories every image in `source/annotated` and `source/annotated_only`, uses every available frozen Notebook 53/53R3 V8.8 corrected page, locks predictions before opening clean truth, benchmarks all clean-matched pages, makes review sheets, and writes a row for every source page. Pages outside the frozen V8.8 roster are reported as `NO_FROZEN_V88_OUTPUT`; they are not silently omitted or mislabelled as V8.8.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q pandas numpy opencv-python-headless matplotlib


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, shutil, zipfile
import cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
ANNOTATED_DIR=ROOT/'source'/'annotated'
ANNOTATED_ONLY_DIR=ROOT/'source'/'annotated_only'
CLEAN_DIR=ROOT/'source'/'clean'
V88_RUN_OVERRIDE=None
V88_ROOTS=[ROOT/'observed_print_classifier_v8_8_notebook53R3',ROOT/'m5_remove_operational_builder_v8_8b',ROOT/'observed_print_classifier_v8_8_notebook53']
OUT=ROOT/'notebook72_v8_8_annotated_only_trial'/(datetime.now().strftime('%Y%m%d_%H%M%S')+'_notebook72')
LOCK=OUT/'01_locked_outputs'; TABLES=OUT/'02_tables'; REVIEW=OUT/'03_performance_review'; AUDIT=OUT/'04_audit'; UPLOAD=OUT/'upload'
for p in [OUT,LOCK,TABLES,REVIEW,AUDIT,UPLOAD]: p.mkdir(parents=True,exist_ok=False)
SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}

def require(x,msg):
    if not bool(x): raise RuntimeError(msg)
def sha(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''): h.update(b)
    return h.hexdigest()
def atomic_json(p,v):
    t=Path(str(p)+'.tmp'); t.write_text(json.dumps(v,indent=2,sort_keys=True,default=str)); t.replace(p)
def read_gray(p):
    a=cv2.imread(str(p),cv2.IMREAD_GRAYSCALE); require(a is not None,f'Unreadable image: {p}'); return a
def image_map(folder):
    out={}; dup={}; folder=Path(folder)
    if not folder.exists(): return out
    for p in sorted(folder.iterdir()):
        if p.is_file() and p.suffix.lower() in SUPPORTED:
            if p.stem in out: dup.setdefault(p.stem,[out[p.stem]]).append(p)
            else: out[p.stem]=p
    require(not dup,f'Duplicate stems in {folder}: {dup}')
    return out
def complete_v88_runs():
    hits=[]
    for root in V88_ROOTS:
        if not root.exists(): continue
        for p in [root]+[x for x in root.rglob('*') if x.is_dir()]:
            if (p/'pages').is_dir() and (p/'experiment_identity.json').exists() and any(q.is_dir() and (q/'corrected.png').exists() for q in (p/'pages').iterdir()): hits.append(p.resolve())
    return sorted({str(p):p for p in hits}.values(),key=lambda p:p.stat().st_mtime)

runs=complete_v88_runs()
V88=Path(V88_RUN_OVERRIDE) if V88_RUN_OVERRIDE else (runs[-1] if runs else None)
require(V88 is not None and (V88/'pages').is_dir(),'No complete V8.8 Notebook 53/53R3 run found; set V88_RUN_OVERRIDE.')

annotated=image_map(ANNOTATED_DIR); annotated_only=image_map(ANNOTATED_ONLY_DIR)
require(annotated or annotated_only,'No source images found.')
require(not (set(annotated)&set(annotated_only)),'Duplicate stems across annotated and annotated_only.')
population=pd.DataFrame([{'page':page,'cohort':cohort,'source_path':str(path),'source_sha256':sha(path)} for cohort,m in [('annotated',annotated),('annotated_only',annotated_only)] for page,path in m.items()]).sort_values(['cohort','page']).reset_index(drop=True)
require(not population.page.duplicated().any(),'Duplicate source page stems.')
population.to_csv(TABLES/'complete_source_population.csv',index=False)
print('SOURCE POPULATION:',len(population),'annotated:',len(annotated),'annotated_only:',len(annotated_only))

vrows=[]
for d in sorted((V88/'pages').iterdir()):
    if not d.is_dir() or not (d/'corrected.png').exists(): continue
    r={'page':d.name,'v88_corrected_path':str(d/'corrected.png'),'v88_corrected_sha256':sha(d/'corrected.png')}
    for n in ['m3.png','safe_remove.png','m5.png','m5_remove.png','observed.png']:
        p=d/n; r[n.replace('.png','_path')]=str(p) if p.exists() else None
    vrows.append(r)
v88=pd.DataFrame(vrows); require(len(v88)>0,'V8.8 pages directory has no corrected outputs.'); require(not v88.page.duplicated().any(),'Duplicate V8.8 pages.')
v88.to_csv(TABLES/'resolved_v88_roster.csv',index=False)

cohort=population.merge(v88,on='page',how='left',validate='one_to_one')
cohort['v88_status']=np.where(cohort.v88_corrected_path.notna(),'FROZEN_V88_OUTPUT_AVAILABLE','NO_FROZEN_V88_OUTPUT')

cohort[
    'locked_corrected_path'
] = None


cohort[
    'lock_sha256'
] = None


cohort[
    'outside_m3_drift'
] = np.nan


cohort[
    'source_equals_v88_observed'
] = False


cohort[
    'source_vs_v88_observed_pixels'
] = np.nan


cohort[
    'v88_observed_path'
] = None


cohort[
    'v88_observed_sha256'
] = None


cohort[
    'v88_m3_sha256'
] = None


cohort[
    'integrity_status'
] = (
    'NOT_RUN_NO_V88_OUTPUT'
)


for i, r in cohort.iterrows():

    if (
        r.v88_status
        != 'FROZEN_V88_OUTPUT_AVAILABLE'
    ):
        continue


    page = str(
        r.page
    )


    # ========================================================
    # V8.8 pixel identity MUST be evaluated against the
    # observed page belonging to the SAME frozen V8.8 run.
    #
    # Do not use source/annotated or source/annotated_only
    # for this immutability assertion.
    # ========================================================

    require(
        isinstance(
            r.get(
                'observed_path'
            ),
            str,
        )
        and r.observed_path,
        (
            f'{page}: frozen V8.8 run does not contain '
            'observed.png'
        ),
    )


    require(
        isinstance(
            r.get(
                'm3_path'
            ),
            str,
        )
        and r.m3_path,
        (
            f'{page}: frozen V8.8 run does not contain '
            'm3.png'
        ),
    )


    v88_observed = read_gray(
        Path(
            r.observed_path
        )
    )


    m3 = (
        read_gray(
            Path(
                r.m3_path
            )
        )
        > 0
    )


    corrected = read_gray(
        Path(
            r.v88_corrected_path
        )
    )


    # ========================================================
    # Shape identity within the V8.8 frozen page artefacts
    # ========================================================

    require(
        (
            v88_observed.shape
            == m3.shape
            == corrected.shape
        ),
        (
            f'{page}: V8.8 observed/M3/corrected '
            'shape mismatch: '
            f'observed={v88_observed.shape}, '
            f'm3={m3.shape}, '
            f'corrected={corrected.shape}'
        ),
    )


    # ========================================================
    # The actual V8.8 immutability assertion
    # ========================================================

    changed = (
        corrected
        != v88_observed
    )


    outside_m3 = (
        changed
        & ~m3
    )


    drift = int(
        outside_m3.sum()
    )


    require(
        drift
        == 0,
        (
            f'{page}: frozen V8.8 corrected page '
            f'changes {drift} pixels outside its '
            'own frozen M3.'
        ),
    )


    # ========================================================
    # Separately examine whether the project source image
    # happens to be identical to the V8.8 observed canvas.
    #
    # This is provenance/registration information.
    # It is NOT the V8.8 immutability test.
    # ========================================================

    project_source = read_gray(
        Path(
            r.source_path
        )
    )


    source_shape_equal = (
        project_source.shape
        == v88_observed.shape
    )


    if source_shape_equal:

        source_vs_v88_observed_pixels = int(
            (
                project_source
                != v88_observed
            ).sum()
        )

        source_equals_v88_observed = bool(
            source_vs_v88_observed_pixels
            == 0
        )

    else:

        source_vs_v88_observed_pixels = np.nan

        source_equals_v88_observed = False


    # ========================================================
    # Lock the actual frozen V8.8 output
    # ========================================================

    dst = (
        LOCK
        / f'{page}.png'
    )


    require(
        cv2.imwrite(
            str(
                dst
            ),
            corrected,
        ),
        (
            f'{page}: failed lock write'
        ),
    )


    cohort.at[
        i,
        'locked_corrected_path'
    ] = str(
        dst
    )


    cohort.at[
        i,
        'lock_sha256'
    ] = sha(
        dst
    )


    cohort.at[
        i,
        'outside_m3_drift'
    ] = drift


    cohort.at[
        i,
        'source_equals_v88_observed'
    ] = (
        source_equals_v88_observed
    )


    cohort.at[
        i,
        'source_vs_v88_observed_pixels'
    ] = (
        source_vs_v88_observed_pixels
    )


    cohort.at[
        i,
        'v88_observed_path'
    ] = str(
        r.observed_path
    )


    cohort.at[
        i,
        'v88_observed_sha256'
    ] = sha(
        Path(
            r.observed_path
        )
    )


    cohort.at[
        i,
        'v88_m3_sha256'
    ] = sha(
        Path(
            r.m3_path
        )
    )


    cohort.at[
        i,
        'integrity_status'
    ] = (
        'PASS_LOCKED'
    )

atomic_json(AUDIT/'prediction_lock.json',lock)

diagnostic_columns = [
    'page',
    'cohort',
    'outside_m3_drift',
    'source_equals_v88_observed',
    'source_vs_v88_observed_pixels',
    'source_path',
    'v88_observed_path',
    'm3_path',
    'v88_corrected_path',
]


display(
    cohort[
        diagnostic_columns
    ]
)


cohort[
    diagnostic_columns
].to_csv(
    AUDIT
    / 'v88_observed_canvas_identity.csv',
    index=False,
)

# Clean truth is first opened here, after the prediction lock.
clean=image_map(CLEAN_DIR); results=[]
def ink(im): return cv2.adaptiveThreshold(im,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0
def f1(ref,pred):
    tp=int(np.sum(ref&pred)); fp=int(np.sum((~ref)&pred)); fn=int(np.sum(ref&(~pred))); p=tp/max(tp+fp,1); r=tp/max(tp+fn,1); return 2*p*r/max(p+r,1e-12),p,r
for _,r in cohort.iterrows():
    page=str(r.page); cp=clean.get(page); row={'page':page,'cohort':r.cohort,'v88_status':r.v88_status,'clean_available':cp is not None}
    if cp is None or r.v88_status!='FROZEN_V88_OUTPUT_AVAILABLE': row.update({'benchmark_status':'NOT_SCORED','whole_page_mae':np.nan,'ink_f1':np.nan,'ink_recall':np.nan})
    else:
        obs=read_gray(Path(r.observed_path)); out=read_gray(Path(r.locked_corrected_path)); truth=read_gray(cp); require(obs.shape==out.shape==truth.shape,f'{page}: clean shape mismatch')
        score,precision,recall=f1(ink(truth),ink(out)); row.update({'benchmark_status':'SCORED_POST_LOCK','whole_page_mae':float(np.abs(out.astype(float)-truth.astype(float)).mean()),'observed_whole_page_mae':float(np.abs(obs.astype(float)-truth.astype(float)).mean()),'ink_f1':score,'ink_precision':precision,'ink_recall':recall,'changed_pixels':int(np.sum(out!=obs))})
    results.append(row)
benchmark=pd.DataFrame(results); benchmark.to_csv(TABLES/'clean_truth_benchmark_all_pages.csv',index=False)

review=cohort[['page','cohort','source_path','v88_status','locked_corrected_path','integrity_status']].merge(benchmark[['page','clean_available','benchmark_status','whole_page_mae','ink_f1','ink_recall']],on='page',validate='one_to_one')
for c in ['performance_rating','annotation_residue','print_damage','musical_meaning_risk','review_notes']: review[c]=''
review['allowed_performance_ratings']='PERFORMANCE_READY|READY_WITH_MINOR_RESIDUE|REVIEW_REQUIRED|UNUSABLE_ANNOTATION_REMAINS|UNUSABLE_PRINT_DAMAGE|NOT_REVIEWABLE_NO_V88_OUTPUT'
review.to_csv(REVIEW/'performance_usability_review.csv',index=False)

covered=cohort[cohort.v88_status=='FROZEN_V88_OUTPUT_AVAILABLE'].sort_values(['cohort','page']).reset_index(drop=True)
sheets=[]
for start in range(0,len(covered),12):
    q=covered.iloc[start:start+12]; fig,ax=plt.subplots(len(q),2,figsize=(14,max(5,5*len(q))),squeeze=False)
    for j,(_,r) in enumerate(q.iterrows()):
        a=read_gray(Path(r.source_path)); b=read_gray(Path(r.locked_corrected_path)); ax[j,0].imshow(a,cmap='gray',vmin=0,vmax=255); ax[j,1].imshow(b,cmap='gray',vmin=0,vmax=255); ax[j,0].set_title(f'{r.page} observed [{r.cohort}]'); ax[j,1].set_title(f'{r.page} V8.8'); ax[j,0].axis('off'); ax[j,1].axis('off')
    plt.tight_layout(); p=REVIEW/f'contact_sheet_{start+1:04d}_{start+len(q):04d}.png'; plt.savefig(p,dpi=110,bbox_inches='tight'); plt.close(fig); sheets.append(p)

uncovered=cohort[cohort.v88_status=='NO_FROZEN_V88_OUTPUT'][['page','cohort','source_path','v88_status']]
uncovered.to_csv(TABLES/'uncovered_pages_requiring_deployable_v88_scorer.csv',index=False)
summary={'created_utc':datetime.now(timezone.utc).isoformat(),'v88_run':str(V88),'source_population_total':int(len(population)),'annotated_pages':len(annotated),'annotated_only_pages':len(annotated_only),'frozen_v88_output_available':int(len(covered)),'no_frozen_v88_output':int(len(uncovered)),'clean_truth_files_discovered':len(clean),'clean_truth_pages_scored_post_lock':int((benchmark.benchmark_status=='SCORED_POST_LOCK').sum()),'performance_review_rows':len(review),'contact_sheets':len(sheets),'outside_m3_drift_total':int(np.nansum(cohort.outside_m3_drift.to_numpy(float))),'truth_firewall':'PASS: clean opened after prediction lock','all_source_pages_accounted_for':bool(len(review)==len(population)),'automatic_unseen_page_v88_inference_performed':False}
atomic_json(OUT/'notebook72_summary.json',summary)
(OUT/'executive_summary.txt').write_text('NOTEBOOK 72 - WHOLE POPULATION\n\n'+'\n'.join(f'{k}: {v}' for k,v in summary.items()))
files=[OUT/'executive_summary.txt',OUT/'notebook72_summary.json',AUDIT/'prediction_lock.json',TABLES/'complete_source_population.csv',TABLES/'resolved_v88_roster.csv',TABLES/'whole_population_v88_coverage.csv',TABLES/'clean_truth_benchmark_all_pages.csv',TABLES/'uncovered_pages_requiring_deployable_v88_scorer.csv',REVIEW/'performance_usability_review.csv']
manifest=[]
for src in files:
    require(src.exists(),f'Missing artefact: {src}'); dst=UPLOAD/src.name; shutil.copy2(src,dst); manifest.append({'file':dst.name,'bytes':dst.stat().st_size,'sha256':sha(dst)})
pd.DataFrame(manifest).to_csv(UPLOAD/'upload_manifest.csv',index=False)
zpath=UPLOAD/'notebook72_upload_bundle.zip'
with zipfile.ZipFile(zpath,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(UPLOAD.iterdir()):
        if p.is_file() and p!=zpath: z.write(p,p.name)
print(json.dumps(summary,indent=2)); print('Review:',REVIEW/'performance_usability_review.csv'); print('Bundle:',zpath)
require(summary['all_source_pages_accounted_for'],'Completeness failure: not every source page is in the final review table.')
